# AURA: EmbeddingGemma 2 on IQ8 / IQ9

Host-to-device runbook for using `google/embeddinggemma-2` as AURA's local embedding model for memory indexing, semantic search, and perception indexing.

Pipeline: Hugging Face model -> ONNX -> QAIRT/QNN converter -> HTP/NPU context binary -> QAIRT runtime -> AURA memory/search index.

No credentials, board passwords, or tokens belong in this notebook.

## Target model and platform

EmbeddingGemma 2 is a unified multimodal embedding model. It maps supported inputs into a 768-dimensional vector space and is intended for local retrieval and semantic similarity workflows.

IQ8/IQ9 is a target platform choice, not a model quantization name. Select the HTP quantization and operator set supported by the installed QAIRT release, then validate embedding similarity and retrieval quality on each board.

References: [EmbeddingGemma 2](https://ai.google.dev/gemma/docs/embeddinggemma), [Sentence Transformers guide](https://ai.google.dev/gemma/docs/embeddinggemma/inference-embeddinggemma-with-sentence-transformers), [Hugging Face model](https://huggingface.co/google/embeddinggemma-2), and [QAIRT overview](https://docs.qualcomm.com/doc/80-63442-10/topic/general_overview.html).

In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
try:
    import truststore
    truststore.inject_into_ssl()
except ImportError:
    print('Optional truststore package is unavailable; use the host CA configuration for Hugging Face downloads.')

import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src' / 'aura').exists():
    PROJECT_ROOT = Path(r'C:/Dropbox/DevRel/Academy/qa-intermediate-curriculum/projects/AURA')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

MODEL_ID = os.getenv('AURA_EMBEDDING_MODEL', 'google/embeddinggemma-2')
TARGET_SOC = os.getenv('AURA_SOC', 'IQ8').upper()  # IQ8 or IQ9
QUANT_PROFILE = os.getenv('AURA_QUANT_PROFILE', 'W8A8')
MAX_LENGTH = int(os.getenv('AURA_EMBEDDING_MAX_LENGTH', '512'))
ARTIFACTS = PROJECT_ROOT / 'artifacts' / 'embeddinggemma2' / TARGET_SOC
ARTIFACTS.mkdir(parents=True, exist_ok=True)
QNN_SDK_ROOT = Path(os.getenv('QNN_SDK_ROOT', r'C:/Qualcomm/AIStack/QAIRT/2.51.40.260930'))
DEVICE_ROOT = os.getenv('AURA_DEVICE_ROOT', '/opt/aura/embeddinggemma2')

RUN_HF_BASELINE = os.getenv('AURA_RUN_HF_BASELINE', 'true').lower() not in {'0', 'false', 'no'}
HF_BASELINE_READY = False
RUN_ONNX_EXPORT = False
RUN_QAIRT_CONVERTER = False
RUN_QAIRT_CONTEXT = False
RUN_DEVICE_COMMAND = False

print({'project': str(PROJECT_ROOT), 'model': MODEL_ID, 'target_soc': TARGET_SOC, 'quant_profile': QUANT_PROFILE, 'artifacts': str(ARTIFACTS)})

## Host prerequisites

Use a Windows x64 CPython environment for the HF baseline. The AURA `.venv` on this machine is ARM64, and PyTorch does not provide a compatible wheel for that interpreter. A working local option is `C:/Users/sreddygu/AppData/Local/Programs/Python/Python311/python.exe`.

Install these on the conversion host, not on a constrained target:

```text
python -m pip install -U torch transformers sentence-transformers optimum onnx onnxruntime numpy
```

If the Hugging Face model requires account access or terms acceptance, complete that interactively. Never put the access token in this notebook.

In [ ]:
if RUN_HF_BASELINE:
    try:
        import numpy as np
        from sentence_transformers import SentenceTransformer
    except ImportError as exc:
        RUN_HF_BASELINE = False
        print(f'HF baseline unavailable: {exc}. Install the host prerequisites, then rerun this cell.')
    else:
        embedding_model = SentenceTransformer(MODEL_ID)
        sample_texts = [
            'A person entered the study room and placed a laptop beside the monitor.',
            'The front camera detected a package near the apartment door.',
            'The microphone heard a request to remember where the laptop was left.',
            'The room is empty; the lights are on and the desk is clear.',
        ]
        sample_vectors = embedding_model.encode(sample_texts, normalize_embeddings=True, convert_to_numpy=True)
        HF_BASELINE_READY = True
        print('embedding shape:', sample_vectors.shape)
        print('embedding dtype:', sample_vectors.dtype)
else:
    print('HF baseline disabled with AURA_RUN_HF_BASELINE=false.')

## Index AURA perception events

AURA keeps the structured event record in SQLite. This notebook stores a normalized embedding matrix beside event metadata so the same event text can be searched with a local dot product. The design preserves AURA's one `Event` contract while allowing EmbeddingGemma 2 to replace a generic vector backend.

Persist the model ID, revision, dimension, normalization rule, and quantization profile with every index. Rebuild the index when any of them change.

In [ ]:
if HF_BASELINE_READY:
    import json
    from aura.core.events import Event
    from aura.memory import CallableEmbeddingBackend, MemoryStore
    events = [
        Event(event='Person entered the study room', type='detection', entities=['person'], location='study room', source='vision'),
        Event(event='Laptop was placed beside the monitor', type='scene', entities=['laptop'], location='study room', source='vision'),
        Event(event='Package detected near the apartment door', type='detection', entities=['package'], location='front door', source='vision'),
        Event(event='User asked where the laptop was left', type='speech', entities=['laptop'], source='speech'),
    ]
    event_texts = [event.event for event in events]
    event_vectors = embedding_model.encode(event_texts, normalize_embeddings=True, convert_to_numpy=True)
    np.savez_compressed(ARTIFACTS / 'event_embeddings.npz', vectors=event_vectors.astype('float32'), ids=np.array([event.id for event in events]))
    (ARTIFACTS / 'index_manifest.json').write_text(json.dumps({'model_id': MODEL_ID, 'target_soc': TARGET_SOC, 'quant_profile': QUANT_PROFILE, 'dimension': int(event_vectors.shape[1]), 'normalized': True, 'count': len(events)}, indent=2), encoding='utf-8')
    embedding_backend = CallableEmbeddingBackend(
        lambda texts: embedding_model.encode(
            list(texts), normalize_embeddings=True, convert_to_numpy=True
        ),
        dimension=int(event_vectors.shape[1]),
    )
    with MemoryStore(
        str(PROJECT_ROOT / 'aura.db'),
        vector_dir=str(ARTIFACTS / 'chroma'),
        embedding_backend=embedding_backend,
    ).connect() as memory:
        for event in events:
            memory.add_event(event)
    print('saved:', ARTIFACTS / 'event_embeddings.npz')
else:
    print('Indexing skipped because the HF baseline is not ready.')

In [ ]:
def semantic_search(query, texts, vectors, model, k=3):
    query_vector = model.encode([query], normalize_embeddings=True, convert_to_numpy=True)[0]
    scores = vectors @ query_vector
    order = scores.argsort()[::-1][:k]
    return [(float(scores[index]), texts[index]) for index in order]

if HF_BASELINE_READY:
    print(*semantic_search('Where did I leave the laptop?', event_texts, event_vectors, embedding_model), sep='\n')
else:
    print('Search skipped because the HF baseline is not ready.')

## Export the HF model to ONNX

The export uses `scripts/export_embeddinggemma_onnx.py`, which wraps the text model with masked mean pooling and L2 normalization so the ONNX output is directly comparable with the host baseline. Inspect the graph before conversion: verify the two inputs, fixed sequence dimensions, output tensor, and 768-D normalized output.

For fixed-shape HTP deployment, a static sequence length such as `MAX_LENGTH=512` is generally easier to validate than fully dynamic shapes.

In [ ]:
onnx_dir = ARTIFACTS / 'onnx'
onnx_dir.mkdir(parents=True, exist_ok=True)
onnx_model = onnx_dir / 'embeddinggemma2_text.onnx'
onnx_command = [sys.executable, str(PROJECT_ROOT / 'scripts' / 'export_embeddinggemma_onnx.py'), '--model', MODEL_ID, '--max-length', str(MAX_LENGTH), '--output', str(onnx_model)]
print(' '.join(shlex.quote(part) for part in onnx_command))
if RUN_ONNX_EXPORT:
    subprocess.run(onnx_command, check=True, cwd=PROJECT_ROOT)
    print('ONNX export complete:', onnx_model)
else:
    print('ONNX export disabled. Review the command, then set RUN_ONNX_EXPORT=True.')

## QAIRT / QNN conversion and HTP context binary

QAIRT's `qairt-converter` converts ONNX into a DLC. The classic QNN path converts the graph into QNN C++ or binary, builds the model library, and uses `qnn-context-binary-generator` to prepare an HTP serialized context. Exact quantization and calibration flags are SDK-release and model dependent, so keep them in `AURA_QAIRT_EXTRA_ARGS` and inspect the installed tool's `--help` before running.

Keep the IQ8/IQ9 choice in the target backend configuration and validation matrix. Record latency, peak memory, output cosine agreement, and top-k recall for both targets.

In [ ]:
onnx_model = next(onnx_dir.glob('*.onnx'), onnx_dir / 'model.onnx')
dlc_model = ARTIFACTS / 'embeddinggemma2.dlc'
qnn_dir = ARTIFACTS / 'qnn'
qnn_dir.mkdir(parents=True, exist_ok=True)
converter = QNN_SDK_ROOT / 'bin' / 'x86_64-windows-msvc' / 'qairt-converter'
if not converter.exists():
    converter = Path('qairt-converter')
extra_args = shlex.split(os.getenv('AURA_QAIRT_EXTRA_ARGS', ''))
converter_command = [str(converter), '--input_network', str(onnx_model), '--output_path', str(dlc_model)] + extra_args
print(' '.join(shlex.quote(part) for part in converter_command))
if RUN_QAIRT_CONVERTER:
    subprocess.run(converter_command, check=True, cwd=PROJECT_ROOT)
    print('DLC created:', dlc_model)
else:
    print('Conversion disabled. Set AURA_QAIRT_EXTRA_ARGS after checking qairt-converter --help.')
context_binary = ARTIFACTS / f'embeddinggemma2_{TARGET_SOC.lower()}.serialized.bin'
cached_dlc = ARTIFACTS / f'embeddinggemma2_{TARGET_SOC.lower()}.cached.dlc'
context_generator = QNN_SDK_ROOT / 'bin' / 'x86_64-windows-msvc' / 'qnn-context-binary-generator.exe'
htp_backend = os.getenv('AURA_QNN_HTP_BACKEND', '<path-to-libQnnHtp.so>')
context_command = [str(context_generator), '--backend', htp_backend, '--dlc_path', str(dlc_model), '--binary_file', str(context_binary), '--output_dlc', str(cached_dlc)]
print('Context command template:')
print(' '.join(shlex.quote(part) for part in context_command))
if RUN_QAIRT_CONTEXT:
    if htp_backend.startswith('<'):
        raise RuntimeError('Set AURA_QNN_HTP_BACKEND before enabling RUN_QAIRT_CONTEXT.')
    subprocess.run(context_command, check=True, cwd=PROJECT_ROOT)
    print('Cached DLC and context binary created:', cached_dlc, context_binary)
print('Runtime command template:')
print('qnn-net-run --backend', shlex.quote(htp_backend), '--input_list inputs.txt --retrieve_context', shlex.quote(str(context_binary)), '--output_dir output')

## QAIRT runtime on IQ8 / IQ9

Copy the serialized context, raw input list, and target-compatible QAIRT runtime libraries to the board. A typical HTP invocation is:

```bash
qnn-net-run \
  --backend /opt/qnn/lib/libQnnHtp.so \
  --input_list inputs.txt \
  --retrieve_context embeddinggemma2_iq8.serialized.bin \
  --output_dir output
```

Use the IQ9 filename and device runtime libraries for IQ9. Do not mix a context binary compiled for one SoC/backend configuration with another. Keep raw tensor layout, dtype, quantization encodings, and output dequantization in the manifest next to the context binary.

In [ ]:
device_command = (f'qnn-net-run --backend {DEVICE_ROOT}/libQnnHtp.so ' f'--input_list {DEVICE_ROOT}/inputs.txt ' f'--retrieve_context {DEVICE_ROOT}/embeddinggemma2_{TARGET_SOC.lower()}.serialized.bin ' f'--output_dir {DEVICE_ROOT}/output')
print('Run on target:', device_command)
print('Use SSH, ADB, or the board deployment wrapper; no credentials are embedded here.')
if RUN_DEVICE_COMMAND:
    raise RuntimeError('Set up the target shell or deployment wrapper before enabling this cell.')

## Validation checklist

1. Compare host FP32 embeddings with QAIRT output after dequantization using cosine similarity.
2. Test a fixed retrieval corpus containing speech transcripts, scene descriptions, object locations, and knowledge chunks.
3. Measure top-1/top-5 retrieval agreement, end-to-end latency, peak memory, context-load time, and sustained indexing throughput.
4. Repeat on IQ8 and IQ9; record QAIRT version, HTP backend, SoC configuration, quantization profile, sequence length, and embedding dimension.
5. Only after numerical and retrieval checks pass, connect the output to AURA's perception -> memory -> reasoning pipeline.

In [ ]:
try:
    import numpy as np
except ImportError:
    print('numpy is not installed. Install the host prerequisites before comparing vectors.')
else:
    def cosine_similarity(left, right):
        left = np.asarray(left, dtype=np.float32).reshape(-1)
        right = np.asarray(right, dtype=np.float32).reshape(-1)
        return float((left @ right) / (np.linalg.norm(left) * np.linalg.norm(right)))

    print('Load one host vector and one dequantized QAIRT vector, then call cosine_similarity(host, qairt).')